# 02 · RLHF & DPO — Learning from Human Preferences
### *Aligning & Deploying LLMs — Unit 2*

Instruction tuning teaches a model to **imitate** good answers, but "good" is hard to write down.
**RLHF** learns a **reward** from human *comparisons* and optimizes the model against it. This notebook has three parts:

1. **Reward model from scratch** — turn rankings into a scalar reward (Bradley–Terry).
2. **The KL leash** — visualize the reward-hacking trade-off from the deck.
3. **DPO from scratch** — align a tiny model straight from preference pairs, no RL loop.

> GPU recommended for Part 3 but not required.

In [ ]:
!pip -q install "transformers>=4.40" torch matplotlib

## Part A · A reward model from comparisons

Each response is described by four features in `[0,1]`: **helpful, concise, safe, polite**.
Humans give **pairwise preferences** (chosen ≻ rejected). We fit a linear reward
`r(x) = w · features(x)` with the **Bradley–Terry** loss `−log σ(r(chosen) − r(rejected))` — exactly the
objective behind real reward models.

In [ ]:
import torch, torch.nn.functional as F

FEATURES = ["helpful", "concise", "safe", "polite"]
responses = {
    "A": torch.tensor([0.9, 0.7, 0.9, 0.8]),  # helpful, fairly concise, safe, polite
    "B": torch.tensor([0.2, 0.9, 0.6, 0.1]),  # blunt "just google it"
    "C": torch.tensor([0.7, 0.15, 0.9, 0.95]), # very polite but rambling
    "D": torch.tensor([0.8, 0.85, 0.85, 0.7]),
}
# Human preferences: (chosen, rejected)
prefs = [("A", "B"), ("A", "C"), ("D", "B"), ("D", "C"), ("A", "D")]

w = torch.zeros(4, requires_grad=True)
opt = torch.optim.Adam([w], lr=0.1)

for step in range(300):
    loss = 0.0
    for c, r in prefs:
        margin = (w @ responses[c]) - (w @ responses[r])
        loss = loss - F.logsigmoid(margin)
    opt.zero_grad(); loss.backward(); opt.step()

w_final = w.detach()
print("Learned reward weights:")
for name, val in zip(FEATURES, w_final):
    print(f"  {name:8s} {val.item():+.2f}")

print("\nReward the model now assigns:")
for k, f in sorted(responses.items(), key=lambda kv: -(w_final @ kv[1]).item()):
    print(f"  {k}: {(w_final @ f).item():+.2f}")

The reward model has distilled messy human rankings into **one number** the optimizer can chase.
Change the `prefs` list to encode a different taste (e.g. reward conciseness) and re-run — the weights follow.

## Part B · The KL leash (reward-hacking trade-off)

PPO maximizes `reward − β · KL(policy ‖ reference)`. With **β too low** the model *reward-hacks* — it exploits
quirks of the reward model and drifts into high-scoring gibberish. With **β too high** it barely moves from the
SFT model. The **net useful gain** peaks in between.

In [ ]:
import numpy as np, matplotlib.pyplot as plt

beta = np.linspace(0.0, 1.0, 200)
chase     = 1 / (1 + beta * 5)      # how hard reward is pursued
coherent  = beta / (beta + 0.12)    # how close we stay to the trusted model
net       = chase * coherent        # useful improvement

plt.figure(figsize=(8, 4.5))
plt.plot(beta, chase,    label="reward chased",   lw=2)
plt.plot(beta, coherent, label="stays coherent",  lw=2)
plt.plot(beta, net,      label="net useful gain",  lw=3)
best = beta[np.argmax(net)]
plt.axvline(best, ls="--", c="gray"); plt.text(best+0.02, 0.05, f"sweet spot β≈{best:.2f}")
plt.xlabel("KL weight  β"); plt.ylabel("relative score"); plt.legend(); plt.title("The KL trade-off in RLHF")
plt.tight_layout(); plt.show()

## Part C · DPO from scratch

**Direct Preference Optimization** skips the separate reward model *and* the RL loop. It optimizes the policy
directly on preference pairs with this loss (β is the implicit KL strength):

$$\mathcal{L}_{DPO} = -\log \sigma\Big(\beta\big[(\log \pi_\theta(y_w|x) - \log \pi_{ref}(y_w|x)) - (\log \pi_\theta(y_l|x) - \log \pi_{ref}(y_l|x))\big]\Big)$$

We implement it directly so it works on any `transformers` version. A frozen **reference** copy anchors the policy.

In [ ]:
import copy, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

name = "distilgpt2"
tk = AutoTokenizer.from_pretrained(name); tk.pad_token = tk.eos_token
policy = AutoModelForCausalLM.from_pretrained(name)
ref = copy.deepcopy(policy).eval()
for p in ref.parameters(): p.requires_grad_(False)

# tiny preference set: prefer the concise, direct answer
data = [
    {"prompt": "Q: How do I reset a router?\nA:",
     "chosen": " Unplug it for 10 seconds, plug it back in, and wait for the lights.",
     "rejected": " Well, routers are complex networking devices with many components and a long history..."},
    {"prompt": "Q: What is the capital of France?\nA:",
     "chosen": " Paris.",
     "rejected": " That is an interesting question about European geography that we could discuss at length."},
    {"prompt": "Q: Is water wet?\nA:",
     "chosen": " Yes.",
     "rejected": " To truly answer we must first define wetness philosophically and physically in detail."},
]

def seq_logprob(model, prompt, completion):
    ids = tk(prompt + completion, return_tensors="pt")
    plen = tk(prompt, return_tensors="pt").input_ids.shape[1]
    out = model(**ids)
    logp = torch.log_softmax(out.logits[:, :-1], dim=-1)
    tgt = ids.input_ids[:, 1:]
    tok_lp = logp.gather(-1, tgt.unsqueeze(-1)).squeeze(-1)[0]
    return tok_lp[plen-1:].sum()   # sum log-prob of the completion tokens

opt = torch.optim.Adam(policy.parameters(), lr=1e-5)
BETA = 0.1
policy.train()
for epoch in range(6):
    total = 0.0
    for ex in data:
        lp_w  = seq_logprob(policy, ex["prompt"], ex["chosen"])
        lp_l  = seq_logprob(policy, ex["prompt"], ex["rejected"])
        with torch.no_grad():
            rp_w = seq_logprob(ref, ex["prompt"], ex["chosen"])
            rp_l = seq_logprob(ref, ex["prompt"], ex["rejected"])
        logits = BETA * ((lp_w - rp_w) - (lp_l - rp_l))
        loss = -torch.nn.functional.logsigmoid(logits)
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item()
    print(f"epoch {epoch}  loss {total/len(data):.4f}")

In [ ]:
# Did the policy learn to prefer the chosen (concise) answers?
policy.eval()
for ex in data:
    with torch.no_grad():
        marg = (seq_logprob(policy, ex["prompt"], ex["chosen"]) -
                seq_logprob(policy, ex["prompt"], ex["rejected"])).item()
    verdict = "prefers CHOSEN ✅" if marg > 0 else "prefers rejected ❌"
    print(f"{verdict}  (margin {marg:+.2f})  |  {ex['prompt'].splitlines()[0]}")

## Recap & your turn

- **Reward model:** turns human *comparisons* into a scalar via the Bradley–Terry loss.
- **KL leash:** the β penalty is what stops reward hacking — there's a sweet spot.
- **DPO:** reaches the same goal with a single classification-style loss and no RL loop — now the default for open models.

**Exercises**
1. In Part A, rewrite `prefs` to reward *conciseness* over *politeness*; watch the weights flip.
2. In Part C, add more preference pairs and raise epochs — does the margin grow?
3. Sweep `BETA` in `[0.01, 0.1, 0.5]` and observe stability.
4. Try the real thing: `pip install trl` and use `DPOTrainer` on a small instruct model.